In [48]:
# Import necessary libraries
import pandas as pd
from pathlib import Path

In [49]:
# Pipe constants
pipe_length = 50.0
pipe_diameter = 0.2

# Sensor positions
NODE_A_X = 5.0
NODE_B_X = 25.0
NODE_C_X = 45.0
tolerance = 0.5

feature_columns = [
    "pressure",
    "velocity-magnitude",
    "turb-kinetic-energy",
    "turb-diss-rate",
    "wall-shear",
    "tailings-vof"
]

raw_data_path = Path.cwd().parent / "data" / "raw"
synthetic_path = Path.cwd().parent / "data" / "synthetic"
output_path = Path.cwd().parent / "data" / "processed"
output_path.mkdir(parents=True, exist_ok=True)

print("CONFIGURATION")
print(f"Node A: {NODE_A_X}, Node B: {NODE_B_X}, Node C: {NODE_C_X}")
print(f"Output path: {output_path}")

CONFIGURATION
Node A: 5.0, Node B: 25.0, Node C: 45.0
Output path: /home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/processed


In [50]:
# Effect factor category
def get_effect_factor(name):
    name = name.lower()

    if "25" in name:
        return 0.25
    elif "50" in name:
        return 0.55
    elif "75" in name:
        return 0.90
    else:
        return 0.0

In [51]:
# Loading the three data files.
scenario_data = []

def load_csv_with_label(path, scenario_id, label):
    df = pd.read_csv(path)
    df["effect_factor"] = get_effect_factor(str(scenario_id))
    df["scenario"] = scenario_id
    df["label"] = label
    return df

# Normal
normal_file = raw_data_path / "normal_dataset.csv"
scenario_data.append((load_csv_with_label(normal_file, "normal_baseline", 0), "normal_baseline", 0))

for file in (synthetic_path / "normal_variations").glob("*.csv"):
    scenario_data.append((load_csv_with_label(file, file.stem, 0), file.stem, 0))

# Leak
for file in (synthetic_path / "leakage_variations").glob("*.csv"):
    scenario_data.append((load_csv_with_label(file, file.stem, 1), file.stem, 1))

# Blockage
for file in (synthetic_path / "blockage_variations").glob("*.csv"):
    scenario_data.append((load_csv_with_label(file, file.stem, 2), file.stem, 2))

# Summary
print("SCENARIO SUMMARY")
print("Total scenarios:", len(scenario_data))
print("Normal:", sum(1 for _,_,l in scenario_data if l==0))
print("Leak:", sum(1 for _,_,l in scenario_data if l==1))
print("Blockage:", sum(1 for _,_,l in scenario_data if l==2))

SCENARIO SUMMARY
Total scenarios: 1
Normal: 1
Leak: 0
Blockage: 0


In [52]:
# Fault type label
def get_fault_type(label):
    mapping = {
        0: "Normal",
        1: "Leak",
        2: "Blockage"
    }
    return mapping.get(label, "Unknown")

In [53]:
# Extracting sensor data points from global dataset
# Finds the closest point to our sensor nodes A,B, C
# Takes measurement from there and saves
def closest_node(df_t, x_target):
    idx = (df_t["x-coordinate"] - x_target).abs().idxmin()
    return df_t.loc[idx]

def extract_sensor_readings(df_scenario, scenario_id, label):
    rows = []

    for t, df_t in df_scenario.groupby("timestep"):

        a = closest_node(df_t, NODE_A_X)
        b = closest_node(df_t, NODE_B_X)
        c = closest_node(df_t, NODE_C_X)

        effect_factor = df_t["effect_factor"].iloc[0]
        fault_type = get_fault_type(label)

        rows.append({
            "scenario_id": scenario_id,
            "timestep": t,

            "node_a_pressure": a["pressure"],
            "velocity_a": a["velocity-magnitude"],
            "tke_a": a["turb-kinetic-energy"],
            "tdr_a": a["turb-diss-rate"],
            "wall_shear_a": a["wall-shear"],
            "tailings_vof_a": a["tailings-vof"],

            "node_b_pressure": b["pressure"],
            "velocity_b": b["velocity-magnitude"],
            "tke_b": b["turb-kinetic-energy"],
            "tdr_b": b["turb-diss-rate"],
            "wall_shear_b": b["wall-shear"],
            "tailings_vof_b": b["tailings-vof"],

            "node_c_pressure": c["pressure"],
            "velocity_c": c["velocity-magnitude"],
            "tke_c": c["turb-kinetic-energy"],
            "tdr_c": c["turb-diss-rate"],
            "wall_shear_c": c["wall-shear"],
            "tailings_vof_c": c["tailings-vof"],

            "label": label,
            "effect_factor": effect_factor,
            "fault_type": fault_type
        })
    df_out = pd.DataFrame(rows)

    print(f"[OK] {scenario_id} → shape {df_out.shape}")

    return df_out

In [54]:
def calculate_derived_features(df_wide):

    df = df_wide.copy()
    # Sort properly
    df = df.sort_values(["scenario_id", "timestep"])

    print("Calculating derived features...")
    # Pressure drops
    df["pressure_drop_ab"] = df["node_a_pressure"] - df["node_b_pressure"]
    df["pressure_drop_bc"] = df["node_b_pressure"] - df["node_c_pressure"]
    df["pressure_drop_ac"] = df["node_a_pressure"] - df["node_c_pressure"]

    # dp/dt per scenario
    def compute_group(group):
        dt = 0.5

        group["dp_dt_a"] = group["node_a_pressure"].diff().fillna(0) / dt
        group["dp_dt_b"] = group["node_b_pressure"].diff().fillna(0) / dt
        group["dp_dt_c"] = group["node_c_pressure"].diff().fillna(0) / dt

        return group

    df = df.groupby("scenario_id", group_keys=False).apply(compute_group)

    # Flow velocity (bulk proxy)
    df["flow_velocity"] = df["velocity_b"]

    # Final cleanup
    df[["dp_dt_a", "dp_dt_b", "dp_dt_c"]] = df[
        ["dp_dt_a", "dp_dt_b", "dp_dt_c"]
    ].fillna(0.0)

    # Debug output
    print("Derived features added successfully")
    print("Final shape:", df.shape)
    print("Columns added:")
    print([
        "pressure_drop_ab",
        "pressure_drop_bc",
        "pressure_drop_ac",
        "dp_dt_a",
        "dp_dt_b",
        "dp_dt_c",
        "flow_velocity"
    ])

    return df

In [55]:
print("EXTRACTING SENSOR READINGS FROM ALL SCENARIOS")
print("=" * 60)

all_extracted = []

total = len(scenario_data)

for i, (df_scenario, scenario_id, label) in enumerate(scenario_data, 1):

    print(f"\n[{i}/{total}] Processing: {scenario_id}")

    extracted = extract_sensor_readings(
        df_scenario,
        scenario_id,
        label
    )

    print(f"  Extracted shape: {extracted.shape}")

    all_extracted.append(extracted)

print("\nCalculating derived features...")

df_all = pd.concat(all_extracted, ignore_index=True)

df_all = calculate_derived_features(df_all)
# Final checks
print("\nFINAL SHAPE:", df_all.shape)

print("\nLABEL DISTRIBUTION:")
print(df_all["label"].value_counts())
# SAVE OUTPUT
output_file = raw_data_path / "sensor_point_dataset_raw.csv"
df_all.to_csv(output_file, index=False)

print("\nDATA SAVED SUCCESSFULLY")
print("Saved to:", output_file)

EXTRACTING SENSOR READINGS FROM ALL SCENARIOS

[1/1] Processing: normal_baseline
[OK] normal_baseline → shape (700, 23)
  Extracted shape: (700, 23)

Calculating derived features...
Calculating derived features...
Derived features added successfully
Final shape: (700, 29)
Columns added:
['pressure_drop_ab', 'pressure_drop_bc', 'pressure_drop_ac', 'dp_dt_a', 'dp_dt_b', 'dp_dt_c', 'flow_velocity']

FINAL SHAPE: (700, 29)

LABEL DISTRIBUTION:
label
0    700
Name: count, dtype: int64

DATA SAVED SUCCESSFULLY
Saved to: /home/darlenewendie/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/raw/sensor_point_dataset_raw.csv


In [56]:
df_all.columns

Index(['timestep', 'node_a_pressure', 'velocity_a', 'tke_a', 'tdr_a',
       'wall_shear_a', 'tailings_vof_a', 'node_b_pressure', 'velocity_b',
       'tke_b', 'tdr_b', 'wall_shear_b', 'tailings_vof_b', 'node_c_pressure',
       'velocity_c', 'tke_c', 'tdr_c', 'wall_shear_c', 'tailings_vof_c',
       'label', 'effect_factor', 'fault_type', 'pressure_drop_ab',
       'pressure_drop_bc', 'pressure_drop_ac', 'dp_dt_a', 'dp_dt_b', 'dp_dt_c',
       'flow_velocity'],
      dtype='str')

In [57]:
df_all.head(20)

,timestep,node_a_pressure,velocity_a,tke_a,tdr_a,wall_shear_a,tailings_vof_a,node_b_pressure,velocity_b,tke_b,...,label,effect_factor,fault_type,pressure_drop_ab,pressure_drop_bc,pressure_drop_ac,dp_dt_a,dp_dt_b,dp_dt_c,flow_velocity
0,1,38814.91760,0.0,0.045525,44.802546,40.826937,0.400001,21136.13553,3.313672,0.035091,...,0,0.0,Normal,17678.78207,19884.954887,37563.736957,0.00000,0.00000,0.000000,3.313672
1,2,38787.16600,0.0,0.045525,44.802546,40.826922,0.400002,21132.73807,3.313717,0.035092,...,0,0.0,Normal,17654.42793,19881.558299,37535.986229,-55.50320,-6.79492,-0.001744,3.313717
2,3,38764.21594,0.0,0.045525,44.802545,40.826917,0.400002,21133.13355,3.313737,0.035092,...,0,0.0,Normal,17631.08239,19881.954589,37513.036979,-45.90012,0.79096,-0.001620,3.313737
3,4,38741.04996,0.0,0.045525,44.802545,40.826915,0.400002,21133.71985,3.313747,0.035093,...,0,0.0,Normal,17607.33011,19882.541067,37489.871177,-46.33196,1.17260,-0.000356,3.313747
4,5,38717.08922,0.0,0.045525,44.802545,40.826915,0.400002,21133.91266,3.313750,0.035093,...,0,0.0,Normal,17583.17656,19882.733871,37465.910431,-47.92148,0.38562,0.000012,3.313750
5,6,38692.79484,0.0,0.045525,44.802545,40.826916,0.400002,21133.99090,3.313751,0.035093,...,0,0.0,Normal,17558.80394,19882.812072,37441.616012,-48.58876,0.15648,0.000078,3.313751
6,7,38668.16490,0.0,0.045525,44.802545,40.826920,0.400002,21133.95409,3.313751,0.035093,...,0,0.0,Normal,17534.21081,19882.775324,37416.986134,-49.25988,-0.07362,-0.000124,3.313751
7,8,38643.64137,0.0,0.045525,44.802546,40.826924,0.400002,21134.03095,3.313749,0.035093,...,0,0.0,Normal,17509.61042,19882.852111,37392.462531,-49.04706,0.15372,0.000146,3.313749
8,9,38618.82501,0.0,0.045525,44.802546,40.826929,0.400001,21134.00057,3.313747,0.035093,...,0,0.0,Normal,17484.82444,19882.821592,37367.646032,-49.63272,-0.06076,0.000278,3.313747
9,10,38594.08079,0.0,0.045525,44.802546,40.826933,0.400001,21134.04651,3.313745,0.035093,...,0,0.0,Normal,17460.03428,19882.867195,37342.901475,-49.48844,0.09188,0.000674,3.313745


In [58]:
print("Label value:", label, "| Type:", type(label))

Label value: 0 | Type: <class 'int'>
